# Inference-noise methods: three-patch comparison

Evaluates one checkpoint on three patches (PHerc0841, PHerc0139 w044, PHerc0343 new letters) under six inference methods.
Run the setup cells first, then run the method cells; run method 0 first so later methods report ratios against the baseline.

Geometry comes from the run config: 4x4 cells of 16px per 64px centre window, so stride 32 already gives 4 windows per cell and stride 16 gives 16.

| method | stride | window weighting | views | aggregation |
|---|---|---|---|---|
| 0 baseline | 32 | uniform | identity | mean |
| 1 smaller stride | 16 | uniform | identity | mean |
| 2 hann | 16 | hann over the 4x4 cell grid | identity | weighted mean |
| 3 tta | 32 | uniform | identity + hflip + rot90 | mean |
| 4 hann + tta | 16 | hann | identity + hflip + rot90 | weighted mean |
| 5 robust | 16 | uniform | identity + hflip + rot90 | per-cell trimmed mean (outer 1/6 each side dropped) |

Each cell prints wall time for the full predict pass (zarr reads included) and signal stats on the 16px output grid:
`ink` = mean score on labelled ink cells, `bg` = mean score on unlabelled papyrus, `rough` = mean |neighbour difference| on background (block/noise proxy).
p343_letters has blank labels, so only the background stats apply there.

In [ ]:
import os
# the only run input: config, architecture, geometry, and output names are derived from it
CHECKPOINT = os.getenv("VESUVIUS_CHECKPOINT", "/vesuvius/models/archs40/holdout_n96_combined_surface_norm_nnpu/final.pth")

In [ ]:
# run selection: everything comes from the checkpoint and the config.json saved beside it
import importlib
import json
import os
import sys
from pathlib import Path

if "/vesuvius" not in sys.path:
    sys.path.insert(0, "/vesuvius")
import utils.config

# reload so a long-lived kernel picks up config.py changes
resolve_run_config = importlib.reload(utils.config).resolve_run_config

MODEL_PATH = Path(CHECKPOINT).resolve()
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"checkpoint is missing: {MODEL_PATH}")
RUN_CONFIG, RUN_CONFIG_PATH = resolve_run_config(MODEL_PATH)
RUN_ID = Path(RUN_CONFIG.get("run_dir") or Path(RUN_CONFIG_PATH).parent).name
_RUN_DATA = RUN_CONFIG["data"]
_RUN_MODEL = RUN_CONFIG["model"]
ARCH = _RUN_MODEL["arch"]
TILE_SIZE = int(_RUN_DATA["tile_size"])
CONTEXT_SIZE = int(_RUN_DATA["context_size"])
CONTEXT_DOWNSAMPLE = int(_RUN_DATA["context_downsample"])
DEPTH = int(_RUN_DATA["depth"])
D_START = int(_RUN_DATA["d_start"])
D_END = int(_RUN_DATA["d_end"])
MULTITILE = bool(_RUN_MODEL["multitile"])
MULTITILE_SUBTILE = int(_RUN_MODEL["multitile_subtile"])
MULTITILE_GRID = int(_RUN_MODEL["multitile_grid"])
TTA = False
TTA_MODE = _RUN_DATA["tta_mode"]
# inference window step in px (<= MULTITILE_GRID * MULTITILE_SUBTILE); 16 = densest, 32 = ~4x faster
EVAL_STRIDE = 32
_RUN_DATA["eval_stride"] = EVAL_STRIDE
print(f"[run] {RUN_ID} | config {RUN_CONFIG_PATH} | checkpoint {MODEL_PATH}")
EVAL_CMAP_NORM = "raw"
EVAL_DPI = 200
OUTPUT_DIR = "output"

EVAL_SCROLLS = {
    "p841": 20260221022814,
    "w044": 20260115000000,
    "p343_letters": 20261007000000,
}

if DEPTH != 8 or not _RUN_MODEL.get("surface_teacher_input") or not _RUN_DATA.get("surface_relative_depth_window"):
    raise RuntimeError("selected run is not literal-surface true slice-8")
if not _RUN_MODEL.get("early_2d_unet") or not _RUN_MODEL.get("gated_stems"):
    raise RuntimeError("selected run is not the early-2D gated baseline")
print(RUN_ID, MODEL_PATH, f"depth={DEPTH}", "literal surface", f"ctx={CONTEXT_SIZE}/ds{CONTEXT_DOWNSAMPLE}", f"stride={EVAL_STRIDE}")

In [ ]:
import gc
import importlib
import os
import sys
import types

REPO = "/vesuvius"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr

from utils.config import Config
from utils.dataloader import _load_unified_cache
from utils.model import create_model
from utils import visualizer as visualizer_module

# reload so an existing kernel cannot retain a stale inference reader
visualizer_module = importlib.reload(visualizer_module)
TBV = visualizer_module.TensorboardVisualizer
predict_tiles = visualizer_module.predict_tiles
group_by_depth = visualizer_module.group_by_depth
imread_gray = visualizer_module.imread_gray

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| bounded-memory visualizer reloaded")

In [ ]:
# config, strict model loading, split helpers, and per-scroll rendering
from utils.config import ScrollConfig
from utils.dataloader import DataManager

_AUXILIARY_PREFIXES = ("supcon_head.", "domain_head.")


def build_config():
    c = Config()
    # replay the full saved run config so geometry, surface window, and splits match training
    for name, value in _RUN_DATA.items():
        if name == "scrolls":
            c.data.scrolls = [ScrollConfig(**item) for item in value]
        elif name != "probe_rois" and hasattr(c.data, name):
            setattr(c.data, name, value)
    c.data.eval_cmap_norm = EVAL_CMAP_NORM
    for name, value in _RUN_MODEL.items():
        if hasattr(c.model, name):
            setattr(c.model, name, value)
    c.model.compile_model = False
    c.tra.supcon = False
    c.tra.dann = False
    c.device = "cuda" if torch.cuda.is_available() else "cpu"
    return c


def _checkpoint_state(path):
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    cleaned = {}
    for key, value in state.items():
        key = key.removeprefix("module.").removeprefix("_orig_mod.")
        if not key.startswith(_AUXILIARY_PREFIXES):
            cleaned[key] = value
    return cleaned, len(state) - len(cleaned)


def load_model(c):
    model, n = create_model(c)
    state, ignored = _checkpoint_state(MODEL_PATH)
    model.load_state_dict(state, strict=True)
    model.eval()
    print(f"loaded {MODEL_PATH} params={n:,} ignored_training_only={ignored}")
    return model


def _scroll_cfg(sid, c):
    return next((scroll for scroll in c.data.scrolls if int(scroll.scroll_id) == int(sid)), None)


def _splits(sid, height, width, c):
    tile = c.data.tile_size
    scroll = _scroll_cfg(sid, c)
    crop_x = scroll.crop_x_frac if scroll else (0.0, 1.0)
    crop_y = scroll.crop_y_frac if scroll else (0.0, 1.0)
    axis = (scroll.split_axis if scroll else "x").lower()
    fraction = scroll.train_split_frac if scroll else 0.75
    x0 = (int(width * crop_x[0]) // tile) * tile
    x1 = max((int(width * crop_x[1]) // tile) * tile, x0 + tile)
    y0 = (int(height * crop_y[0]) // tile) * tile
    y1 = max((int(height * crop_y[1]) // tile) * tile, y0 + tile)
    if axis == "y":
        split_end = y0 + (int((y1 - y0) * fraction) // tile) * tile
    else:
        split_end = x0 + (int((x1 - x0) * fraction) // tile) * tile
    # render the full mask frame; the split line is measured from the frame origin
    return (0, height), (0, width), split_end // tile, axis


def _manual_train_grid(sid, full_y, full_x, grid_shape, c):
    """train cells from train_masks/<sid>.png, matching the training-time eval figure"""
    if bool(getattr(c.data, "simple_split", True)):
        return None
    train_mask = imread_gray(os.path.join(c.data.train_mask_dir, f"{sid}.png"))
    if train_mask is None:
        return None
    unit = int(c.model.multitile_subtile)
    aligned = DataManager._align_manual_mask(train_mask, unit)
    height, width = grid_shape
    region = aligned[full_y[0]:full_y[0] + height * unit, full_x[0]:full_x[0] + width * unit]
    region = np.pad(region, ((0, height * unit - region.shape[0]), (0, width * unit - region.shape[1])))
    return region.reshape(height, unit, width, unit).all(axis=(1, 3))


def get_norm(sid, vol, mask):
    stats = _load_unified_cache().get(str(sid))
    if stats and all(key in stats for key in ("mean", "std", "min", "max")):
        return stats["mean"], stats["std"], stats["min"], stats["max"]
    raise RuntimeError(f"normalization cache missing for {sid}")


def _run_name():
    return RUN_ID


def _load_labels(sid, label_dir, shape):
    """run's label dir, then ./inklabels, then an empty label so inference still renders"""
    for directory in dict.fromkeys((label_dir, "./inklabels")):
        image = imread_gray(os.path.join(directory, f"{sid}.png"))
        if image is not None:
            if directory != label_dir:
                print(f"[warn] no label for {sid} in {label_dir}; using {directory}")
            return (image > 127).astype(np.uint8), os.path.basename(os.path.normpath(directory))
    print(f"[warn] no inklabel for {sid} in {label_dir} or ./inklabels; overlay will be empty")
    return np.zeros(shape, dtype=np.uint8), "none"

In [ ]:
# build config and load the model once
from utils.platform import get_zarr_dir

C = build_config()
C.data.zarr_path = get_zarr_dir()
MODEL = load_model(C)
print(
    "arch:", C.model.arch,
    "| zarr:", C.data.zarr_path,
    "| context:", C.data.context_size,
    "ds", C.data.context_downsample,
    "| feature_attn_mil:", C.model.feature_attn_mil,
    "| learned_surface:", C.model.learned_surface,
    "| new_surface:", C.model.new_learned_surface,
    "| multitile:", f"{C.model.multitile_subtile}x{C.model.multitile_grid}",
)

In [ ]:
# norm helper override: honor future run-config norm modes, including surface-anchor
from utils.norm import UNIFIED_CACHE_PATH, compute_norm, ensure_surface_anchors, load_cached_norm

def get_norm(sid, vol, mask):
    del vol, mask
    mode = str(getattr(C.data, "norm_mode", "global"))
    if mode == "surface_anchor":
        ensure_surface_anchors([sid], C.data.zarr_path, surface_dir=C.data.surface_label_dir)
    stats = load_cached_norm(sid, UNIFIED_CACHE_PATH, mode=mode)
    if stats is None:
        compute_norm(
            sid,
            C.data.zarr_path,
            UNIFIED_CACHE_PATH,
            mask_dir=getattr(C.data, "mask_dir", "./masks"),
        )
        stats = load_cached_norm(sid, UNIFIED_CACHE_PATH, mode=mode)
    if stats is None:
        raise RuntimeError(f"normalization cache missing for {sid} (mode={mode})")
    return stats

In [ ]:
# method machinery: custom window accumulation patched into the shared predict_tiles
import time
import numpy as np
import torch
from IPython.display import display
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

METHODS = {
    "m0_baseline": dict(title="0 baseline", stride=32, hann=False, views=["id"], trim=False),
    "m1_stride16": dict(title="1 stride 16", stride=16, hann=False, views=["id"], trim=False),
    "m2_hann16": dict(title="2 hann, stride 16", stride=16, hann=True, views=["id"], trim=False),
    "m3_tta": dict(title="3 tta (id+hflip+rot90)", stride=32, hann=False, views=["id", "hflip", "rot90"], trim=False),
    "m4_hann16_tta": dict(title="4 hann stride 16 + tta", stride=16, hann=True, views=["id", "hflip", "rot90"], trim=False),
    "m5_trimmed16_tta": dict(title="5 stride 16 + tta, trimmed mean", stride=16, hann=False, views=["id", "hflip", "rot90"], trim=True),
}
TRIM_DIVISOR = 6
ACTIVE = {}
_STACK = {}
RESULTS = {}
_LABEL_CACHE = {}

_VIEW_OPS = {
    "id": (lambda t: t, lambda g: g),
    "hflip": (lambda t: torch.flip(t, dims=[-1]), lambda g: torch.flip(g, dims=[2])),
    "rot90": (lambda t: torch.rot90(t, 1, dims=[-2, -1]), lambda g: torch.rot90(g, -1, dims=[1, 2])),
}


def _hann_grid(n):
    # cell-centre samples never reach zero; overlapping windows at one-cell stride sum to a constant
    w = np.sin(np.pi * (np.arange(n) + 0.5) / n) ** 2
    return np.outer(w, w).astype(np.float32)


def _scatter_window(grid, y_off, x_off, off, out_tile, h_small, w_small, weight, s_sum, s_cnt, stack_k):
    n = grid.shape[0]
    by = (y_off + off) // out_tile
    bx = (x_off + off) // out_tile
    y0, y1 = max(by, 0), min(by + n, h_small)
    x0, x1 = max(bx, 0), min(bx + n, w_small)
    if y0 >= y1 or x0 >= x1:
        return
    g = grid[y0 - by:y1 - by, x0 - bx:x1 - bx]
    w = weight[y0 - by:y1 - by, x0 - bx:x1 - bx]
    s_sum[y0:y1, x0:x1] += g * w
    s_cnt[y0:y1, x0:x1] += w
    if stack_k:
        vals, idx = _STACK["vals"], _STACK["idx"]
        slots = idx[y0:y1, x0:x1]
        ok = slots < stack_k
        yy = np.broadcast_to(np.arange(y0, y1)[:, None], slots.shape)[ok]
        xx = np.broadcast_to(np.arange(x0, x1)[None, :], slots.shape)[ok]
        vals[slots[ok], yy, xx] = g[ok]
        idx[y0:y1, x0:x1] += ok


def _method_chunk(valid, pm_sum, pm_cnt, pm_sum_t, pm_cnt_t, model, device,
                  tile, out_tile, mt_n, h_small, w_small, tf, tf_inv,
                  infer_bs, also_tta, tta):
    """drop-in for visualizer._process_chunk: views, window weights and per-cell stacks come from ACTIVE."""
    views = [_VIEW_OPS[name] for name in ACTIVE["views"]]
    weight = _hann_grid(mt_n) if ACTIVE["hann"] else np.ones((mt_n, mt_n), np.float32)
    off = (tile - mt_n * out_tile) // 2
    stack_k = 0
    if ACTIVE["trim"]:
        stack_k = ((mt_n * out_tile) // ACTIVE["stride"]) ** 2 * len(views) + 4 * len(views)
        if "vals" not in _STACK:
            _STACK["vals"] = np.full((stack_k, h_small, w_small), np.nan, dtype=np.float16)
            _STACK["idx"] = np.zeros((h_small, w_small), dtype=np.int16)
    forward = getattr(model, "_eager_forward_with_extras", None)

    def infer(x, depth, confidence):
        kwargs = {"teacher_surface_depth": depth, "teacher_surface_confidence": confidence}
        return forward(x, **kwargs)[0] if forward is not None else model(x, **kwargs)

    with torch.no_grad(), torch.autocast(device_type=device, enabled=device == "cuda"):
        for i in range(0, len(valid), infer_bs):
            chunk = valid[i:i + infer_bs]
            coords = [(item[1], item[2]) for item in chunk]
            bt = torch.from_numpy(np.stack([item[0] for item in chunk])).float().unsqueeze(1).to(device)
            depth = torch.from_numpy(np.stack([item[3] for item in chunk])).float().unsqueeze(1).to(device)
            confidence = torch.from_numpy(np.stack([item[4] for item in chunk])).float().unsqueeze(1).to(device)
            batch = bt.shape[0]
            grids = []
            for op, inverse in views:
                g = torch.sigmoid(infer(op(bt).contiguous(), op(depth).contiguous(), op(confidence).contiguous()))
                grids.append(inverse(g.view(batch, mt_n, mt_n)).float().cpu().numpy())
            if stack_k:
                for k, (y_off, x_off) in enumerate(coords):
                    for g in grids:
                        _scatter_window(g[k], y_off, x_off, off, out_tile, h_small, w_small, weight, pm_sum, pm_cnt, stack_k)
            else:
                mean_grid = np.mean(grids, axis=0)
                for k, (y_off, x_off) in enumerate(coords):
                    _scatter_window(mean_grid[k], y_off, x_off, off, out_tile, h_small, w_small, weight, pm_sum, pm_cnt, 0)
            del bt, depth, confidence


visualizer_module._process_chunk = _method_chunk


def _trimmed_mean():
    vals, idx = _STACK["vals"], _STACK["idx"]
    depth, height, width = vals.shape
    out = np.full((height, width), np.nan, dtype=np.float32)
    ranks = np.arange(depth)[:, None, None]
    for r0 in range(0, height, 256):
        block = vals[:, r0:r0 + 256].astype(np.float32)
        block.sort(axis=0)  # unfilled slots are NaN and sort last
        count = np.minimum(idx[r0:r0 + 256], depth).astype(np.int32)[None]
        trim = count // TRIM_DIVISOR
        keep = (ranks >= trim) & (ranks < count - trim)
        kept = keep.sum(axis=0)
        total = np.where(keep, block, 0.0).sum(axis=0)
        out[r0:r0 + 256] = np.where(kept > 0, total / np.maximum(kept, 1), np.nan)
    return out


def _cell_labels(sid, labels, mask, out_tile, pred_shape):
    if sid not in _LABEL_CACHE:
        h, w = pred_shape
        crop = (slice(0, h * out_tile), slice(0, w * out_tile))
        shape = (h, out_tile, w, out_tile)
        _LABEL_CACHE[sid] = (
            labels[crop].reshape(shape).any(axis=(1, 3)),
            mask[crop].reshape(shape).all(axis=(1, 3)),
        )
    return _LABEL_CACHE[sid]


def _signal_stats(pred, ink_cells, papyrus):
    finite = np.isfinite(pred)
    ink = finite & papyrus & ink_cells
    bg = finite & papyrus & ~ink_cells
    both_x = bg[:, 1:] & bg[:, :-1]
    both_y = bg[1:] & bg[:-1]
    steps = np.concatenate([
        np.abs(pred[:, 1:] - pred[:, :-1])[both_x],
        np.abs(pred[1:] - pred[:-1])[both_y],
    ])
    return {
        "ink": float(pred[ink].mean()) if ink.any() else float("nan"),
        "bg": float(pred[bg].mean()) if bg.any() else float("nan"),
        "bg_std": float(pred[bg].std()) if bg.any() else float("nan"),
        "rough": float(steps.mean()) if steps.size else float("nan"),
    }


def _pct(value, base):
    return f"{100.0 * (value / base - 1.0):+.0f}%" if base and np.isfinite(value) and np.isfinite(base) else "n/a"


def run_method(method, name, sid, model, c):
    cfg = METHODS[method]
    ACTIVE.clear()
    ACTIVE.update(cfg)
    _STACK.clear()
    c.data.eval_stride = cfg["stride"]
    sid = int(sid)
    fake = types.SimpleNamespace(c=c)
    fake._display_norm = types.MethodType(TBV._display_norm, fake)
    vol = zarr.open(os.path.join(c.data.zarr_path, f"{sid}.zarr"), mode="r")
    height, width = int(vol.shape[1]), int(vol.shape[2])
    mask_image = imread_gray(f"masks/{sid}.png")
    if mask_image is None:
        print(f"[skip] {name} ({sid}): masks/{sid}.png is missing")
        return
    mask = (mask_image > 127).astype(np.uint8)
    label_dir = getattr(c.data, "inklabel_dir", None) or "./eroded_inklabels"
    labels, label_name = _load_labels(sid, label_dir, mask.shape)
    common_height = min(height, mask.shape[0], labels.shape[0])
    common_width = min(width, mask.shape[1], labels.shape[1])
    vol = vol[:, :common_height, :common_width]
    mask = mask[:common_height, :common_width]
    labels = labels[:common_height, :common_width]
    mean, std, global_min, global_max = get_norm(sid, vol, mask)
    surface_dir = os.path.join(c.data.surface_label_dir, str(sid))
    surface_depth = np.load(os.path.join(surface_dir, "depth.npy"), mmap_mode="r")
    surface_confidence = np.load(os.path.join(surface_dir, "confidence.npy"), mmap_mode="r")

    full_y, full_x, _, _ = _splits(sid, common_height, common_width, c)
    coords = TBV._gen_tile_coords(fake, (c.data.d_start, c.data.d_end), full_y, full_x, mask, z_step=c.data.depth)
    grouped = group_by_depth(coords)
    if len(grouped) != 1:
        raise RuntimeError(f"surface-relative inference expected one depth pass, got {len(grouped)}")
    depth_offset = next(iter(grouped))

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    pred = predict_tiles(
        c, model, vol, mask, grouped[depth_offset], full_y, full_x,
        c.data.d_start + depth_offset, f"{method}_{sid}", mean, std, global_min, global_max,
        surface_depth_map=surface_depth, surface_confidence_map=surface_confidence,
    )
    if cfg["trim"]:
        pred = _trimmed_mean()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    _STACK.clear()

    out_tile = int(c.model.multitile_subtile)
    ink_cells, papyrus = _cell_labels(sid, labels, mask, out_tile, pred.shape)
    stats = _signal_stats(pred, ink_cells, papyrus)
    stats["seconds"] = seconds
    RESULTS[(name, method)] = stats
    base = RESULTS.get((name, "m0_baseline"))
    line = f"[{method}] {name}: {seconds:7.1f}s"
    if base:
        line += f" ({seconds / base['seconds']:.1f}x)"
    line += f" | ink {stats['ink']:.3f} ({_pct(stats['ink'], base['ink']) if base else 'n/a'})"
    line += f" | bg {stats['bg']:.3f} ({_pct(stats['bg'], base['bg']) if base else 'n/a'})"
    line += f" | bg_std {stats['bg_std']:.3f} ({_pct(stats['bg_std'], base['bg_std']) if base else 'n/a'})"
    line += f" | rough {stats['rough']:.4f} ({_pct(stats['rough'], base['rough']) if base else 'n/a'})"
    print(line)

    manual_train_grid = _manual_train_grid(sid, full_y, full_x, pred.shape, c)

    def _raw(subdir):
        image = imread_gray(f"./inklabels/{subdir}/{sid}.png")
        if image is None:
            return None
        mh, mw = mask.shape
        rh, rw = image.shape
        crop = image[
            int(full_y[0] * rh / mh):int(full_y[1] * rh / mh),
            int(full_x[0] * rw / mw):int(full_x[1] * rw / mw),
        ]
        return cv2.resize(crop, (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_LINEAR)

    fig = TBV._create_eval_figure_2x2(
        fake, pred, None, ink_cells, _raw("1_1um"), _raw("2_4um"), None, None, manual_train_grid,
    )
    fig.suptitle(f"{name} (scroll {sid}) | {cfg['title']} | {seconds:.1f}s | labels {label_name}", fontsize=10)
    fig_w, fig_h = fig.get_size_inches()
    pos = fig.axes[0].get_position()
    factor = max(1.0, common_width / (pos.width * fig_w * EVAL_DPI), common_height / (pos.height * fig_h * EVAL_DPI))
    fig.set_size_inches(fig_w * factor, fig_h * factor)
    for text in fig.findobj(plt.Text):
        text.set_fontsize(text.get_fontsize() * factor)
    for line_artist in fig.findobj(plt.Line2D):
        line_artist.set_linewidth(line_artist.get_linewidth() * factor)
    out_dir = os.path.join(OUTPUT_DIR, "eval_visualizations", RUN_ID, "methods", method)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}_{sid}.png")
    fig.savefig(out_path, dpi=EVAL_DPI, bbox_inches="tight")
    plt.close(fig)
    np.save(os.path.join(out_dir, f"{name}_{sid}_pred.npy"), pred)
    del vol, mask, labels, fig
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    preview = Image.open(out_path).convert("RGB")
    preview.thumbnail((2400, 2400))
    display(preview)
    print(f"[saved] {out_path} pred grid {pred.shape}")

## Method 0: baseline (stride 32, single pass)

In [ ]:
# PHerc0841
run_method("m0_baseline", "p841", EVAL_SCROLLS["p841"], MODEL, C)

In [ ]:
# PHerc0139 w044
run_method("m0_baseline", "w044", EVAL_SCROLLS["w044"], MODEL, C)

In [ ]:
# PHerc0343 new letters (not 343p)
run_method("m0_baseline", "p343_letters", EVAL_SCROLLS["p343_letters"], MODEL, C)

## Method 1: smaller stride (16), uniform average

In [ ]:
# PHerc0841
run_method("m1_stride16", "p841", EVAL_SCROLLS["p841"], MODEL, C)

In [ ]:
# PHerc0139 w044
run_method("m1_stride16", "w044", EVAL_SCROLLS["w044"], MODEL, C)

In [ ]:
# PHerc0343 new letters (not 343p)
run_method("m1_stride16", "p343_letters", EVAL_SCROLLS["p343_letters"], MODEL, C)

## Method 2: stride 16 with hann-weighted averaging

In [ ]:
# PHerc0841
run_method("m2_hann16", "p841", EVAL_SCROLLS["p841"], MODEL, C)

In [ ]:
# PHerc0139 w044
run_method("m2_hann16", "w044", EVAL_SCROLLS["w044"], MODEL, C)

In [ ]:
# PHerc0343 new letters (not 343p)
run_method("m2_hann16", "p343_letters", EVAL_SCROLLS["p343_letters"], MODEL, C)

## Method 3: TTA (identity + hflip + rot90), stride 32

In [ ]:
# PHerc0841
run_method("m3_tta", "p841", EVAL_SCROLLS["p841"], MODEL, C)

In [ ]:
# PHerc0139 w044
run_method("m3_tta", "w044", EVAL_SCROLLS["w044"], MODEL, C)

In [ ]:
# PHerc0343 new letters (not 343p)
run_method("m3_tta", "p343_letters", EVAL_SCROLLS["p343_letters"], MODEL, C)

## Method 4: hann stride 16 + TTA

In [ ]:
# PHerc0841
run_method("m4_hann16_tta", "p841", EVAL_SCROLLS["p841"], MODEL, C)

In [ ]:
# PHerc0139 w044
run_method("m4_hann16_tta", "w044", EVAL_SCROLLS["w044"], MODEL, C)

In [ ]:
# PHerc0343 new letters (not 343p)
run_method("m4_hann16_tta", "p343_letters", EVAL_SCROLLS["p343_letters"], MODEL, C)

## Method 5: stride 16 + TTA with per-cell trimmed mean (outlier rejection)

In [ ]:
# PHerc0841
run_method("m5_trimmed16_tta", "p841", EVAL_SCROLLS["p841"], MODEL, C)

In [ ]:
# PHerc0139 w044
run_method("m5_trimmed16_tta", "w044", EVAL_SCROLLS["w044"], MODEL, C)

In [ ]:
# PHerc0343 new letters (not 343p)
run_method("m5_trimmed16_tta", "p343_letters", EVAL_SCROLLS["p343_letters"], MODEL, C)

: 